# Phonological Distance Analysis

Explores how measuring acoustic distance in XLS-R 300M embedding space can help provide insight into audio language model biases. 

**Pipeline stages** (run once each, then reuse outputs):
1. Extract 16kHz mono WAVs from the HF dataset
2. Transcribe with Whisper large-v3 (word-level timestamps)
3. Compute per-word DTW distances vs. American English reference speakers
4. Analysis & figures — see [`Figures_Phonolgical_Distance.ipynb`](Figures_Phonolgical_Distance.ipynb)

`audio → Whisper word timestamps → crop each word → XLS-R layer-14 frame embeddings → DTW-align to American-English utterances of the same word → mean cosine = per-word distance`

**Method reference.** The XLS‑R + dynamic time warping (DTW) acoustic‑distance measure follows Bartelds et al. (2022), *Neural representations for modeling variation in speech* (Journal of Phonetics 92:101137, [doi:10.1016/j.wocn.2022.101137](https://doi.org/10.1016/j.wocn.2022.101137)):

```bibtex
@article{bartelds2022neural,
  author  = {Bartelds, Martijn and de Vries, Wietse and Sanal, Faraz and Richter, Caitlin and Liberman, Mark and Wieling, Martijn},
  title   = {Neural representations for modeling variation in speech},
  journal = {Journal of Phonetics},
  volume  = {92},
  pages   = {101137},
  year    = {2022},
  doi     = {10.1016/j.wocn.2022.101137},
}
```

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, re

ROOT  = Path(os.getcwd()).parent
OUT   = ROOT / "results" / "phonological_distance"
DIST  = OUT  / "distances" / "word_distances.csv"

ACCENT_ORDER  = ["Nigerian", "Chinese", "Indian", "British", "American"]
ACCENT_COLORS = {
    "Nigerian": "#d62728",
    "Chinese":  "#ff7f0e",
    "Indian":   "#2ca02c",
    "British":  "#1f77b4",
    "American": "#9467bd",
}

## Stage 1 — Extract WAV files
Downloads the gated human-speech dataset from Hugging Face and writes 16 kHz mono WAVs.

In [ ]:
# Run once — skips if wav files already exist
!cd .. && python src/phonological_distance_pipeline.py --stage extract

In [ ]:
# Verify: show speaker manifest
with open(OUT / "speakers.json") as f:
    speakers = json.load(f)

df_sp = pd.DataFrame(speakers)
print(df_sp["accent"].value_counts().to_string())
print(f"\nTotal: {len(df_sp)} speakers")

## Stage 2 — Transcribe with Whisper

Uses `openai/whisper-large-v3` via the HuggingFace pipeline with `return_timestamps='word'`.
Output: per-speaker TSV files with `word \t start \t end` (filtered to script vocabulary).

> **Spot-check** a few Nigerian and Chinese speakers after this runs to verify timestamps look reasonable.

In [ ]:
# Skips already-transcribed speakers
!cd .. && python src/phonological_distance_pipeline.py --stage transcribe

In [ ]:
# Spot-check: show word coverage per speaker
import csv
from pathlib import Path

ts_dir = OUT / "timestamps"
rows = []
for p in sorted(ts_dir.glob("*.tsv")):
    with open(p, newline="") as f:
        n = sum(1 for _ in csv.DictReader(f, delimiter="\t"))
    sp_id = p.stem
    accent = next((s["accent"] for s in speakers if s["speaker_id"] == sp_id), "?")
    rows.append({"speaker": sp_id, "accent": accent, "n_words": n})

df_ts = pd.DataFrame(rows).sort_values(["accent","speaker"])
print(df_ts.to_string(index=False))
print("\nMean words by accent:")
print(df_ts.groupby("accent")["n_words"].agg(["mean","min","max"]).round(1))

In [ ]:
# Show first few words for a Nigerian and Chinese speaker
for target_accent in ["Nigerian", "Chinese"]:
    example = next(s for s in speakers if s["accent"] == target_accent)
    ts_path = ts_dir / f"{example['speaker_id']}.tsv"
    print(f"\n{example['speaker_id']} ({target_accent}):")
    with open(ts_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f, delimiter="\t")):
            print(f"  {row['word']:20s}  {row['start']}–{row['end']}")
            if i >= 14: break

## Stage 3 — Acoustic Distance Computation

For each target speaker × word, extracts XLS-R 300M hidden states at layer 14,
then computes DTW cosine distance against every American English reference speaker
who produced that same word. Saves `results/phonological_distance/distances/word_distances.csv`.

_Why an embedding distance reflects pronunciation:_ following Bartelds et al. (2022), two utterances of the same word are compared by **DTW-aligning their XLS-R frame embeddings**, and the mean aligned cosine distance tracks human-perceived pronunciation difference. We take that distance against the American-English references; layer 14 is the choice justified by the ablation in `src/run_layer_ablation.sh`.

> **Runtime note**: ~10–20 minutes on Apple MPS (with CPU fallback for weight_norm). The DTW inner loop runs on CPU; segments are short (~0.2–0.8s) so this is fast.

In [ ]:
# Rerun to regenerate; already idempotent (overwrites CSV)
!cd .. && python src/phonological_distance_pipeline.py --stage distances --layer 14

## Output

Stage 3 writes `results/phonological_distance/distances/word_distances.csv` — one row per *(speaker, word)* with its DTW cosine distance from the American-English references. A quick peek at the shipped, duration-normalized output is below.

**The figures and statistical analysis** — acoustic-distance ↔ delivery-score correlations, the phonological-feature breakdown, and the layer ablation — **live in [`Figures_Phonolgical_Distance.ipynb`](Figures_Phonolgical_Distance.ipynb).** This notebook is just the data-generation pipeline.

In [ ]:
import pandas as pd
# Shipped example: the duration-normalized per-word distances used by the figures notebook.
df = pd.read_csv(ROOT / 'results' / 'phone_distance_distances_only' / 'human' / 'word_distances_personal_intro.csv')
print(f"{len(df):,} rows | {df['speaker_id'].nunique()} speakers | {df['word'].nunique()} words")
print(df.groupby('accent')['speaker_id'].nunique().to_string())
df.head()